# Integridad

Verifica versiones, hashes, intentos y participantes. Los datos ausentes permanecen desconocidos; ningún contador se estima desde caracteres.

In [ ]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from native_eval.bundle import COLUMNS, comparison, verify_bundle
bundle = Path(globals().get('BUNDLE', os.environ.get('NATIVE_EVAL_BUNDLE', '.runs/bundle')))
runs = pd.read_csv(bundle / 'runs.csv') if (bundle / 'runs.csv').is_file() else pd.DataFrame(columns=COLUMNS)
print('Sin resultados: no hay corridas de evaluación.' if runs.empty else f'{len(runs)} intentos observados; se muestran también los fallos.')


In [ ]:
if (bundle / 'manifest.json').exists():
    manifest = verify_bundle(bundle)
    campaign = json.loads((bundle / 'campaign.json').read_text())
    display({k: campaign[k] for k in ('protocol','benchmark_commit','harbor','claude_code','model','effort','seed')})
    print(f"{len(manifest['files'])} archivos verificados")
    expected = {s['slot_id'] for s in campaign['schedule']}
    observed = set(runs['slot_id'])
    display({'planned':len(expected), 'observed':len(observed), 'missing':sorted(expected-observed), 'unexpected':sorted(observed-expected)})
if not runs.empty:
    display(runs[['slot_id','accounting_complete','protocol_ok','reward','success','incident_count']])
    calls = pd.DataFrame(json.loads((bundle / 'calls.json').read_text()))
    display(calls)
    display(pd.DataFrame(json.loads((bundle / 'incidents.json').read_text())))


# Comparación descriptiva por tarea

Dos tareas y tres repeticiones por sistema: no se afirma superioridad general, equivalencia ni mecanismos causales. Costo incluye fallos; costo por éxito es indefinido sin éxitos o si falta consumo.

In [ ]:
if not runs.empty:
    display(runs)
    display(comparison(runs))
    for task, group in runs.groupby('task'):
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
        for axis, metric in zip(axes, ['input_tokens','model_steps','wall_seconds']):
            for arm, frame in group.groupby('arm'):
                axis.scatter(frame['repetition'], frame[metric], label=arm)
            axis.set(xlabel='Repetición', ylabel=metric, title=task)
            axis.legend()
        plt.tight_layout(); plt.show()
    pairs = runs.pivot(index=['task','repetition'], columns='arm', values=['success','wall_seconds'])
    if all(('success', a) in pairs.columns for a in ['agentplat','agent-teams']):
        matched = pairs.loc[pairs[('success','agentplat')].eq(True) & pairs[('success','agent-teams')].eq(True)]
        display(matched)
        print('Tiempo pareado: solo pares con éxito en ambos sistemas. La tabla completa anterior conserva timeouts y fallos.')


# Coordinación observada

Eventos operativos depurados: participantes, herramientas, destinatarios y tiempos. Los textos y registros originales quedan con el ejecutor. Una diferencia de coordinación no identifica su efecto causal.

In [ ]:
events = pd.DataFrame(json.loads((bundle / 'events.json').read_text())) if (bundle / 'events.json').is_file() else pd.DataFrame()
if not events.empty:
    display(events)
    for slot, frame in events.groupby('slot_id'):
        visible = frame.dropna(subset=['actor','time']).copy()
        if visible.empty: continue
        visible['elapsed_seconds'] = visible['time'] - frame['time'].min()
        fig, axis = plt.subplots(figsize=(10, 3))
        for actor, participant in visible.groupby('actor'):
            axis.scatter(participant['elapsed_seconds'], [actor]*len(participant), label=actor, s=15)
        axis.set(title=slot, xlabel='Segundos desde el primer evento', ylabel='Participante')
        plt.tight_layout(); plt.show()
        display(frame.groupby(['actor','operation'], dropna=False).size().rename('events').reset_index())
